# E-Commerce Customer Intelligence

This notebook is the narrative analysis layer for the portfolio project. It queries the SQLite warehouse, validates headline KPIs, and turns channel, funnel, and retention findings into business decisions. The dataset is synthetic and reproducible (seed 42).

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data' / 'warehouse' / 'ecommerce_analytics.sqlite'
PROCESSED = ROOT / 'data' / 'processed'

## 1. Headline KPI reconciliation

In [ ]:
kpis = json.loads((ROOT / 'reports' / 'kpi_summary.json').read_text())
pd.Series(kpis, name='value').to_frame()

The executive view separates gross demand from retained net revenue. AOV uses completed orders, while conversion uses all sessions; that denominator discipline prevents an inflated funnel result.

## 2. Acquisition quality: 90-day value

In [ ]:
channel_value = pd.read_csv(PROCESSED / 'channel_90d_value.csv')
channel_value.style.format({
    'revenue_90d': '€{:,.0f}',
    'revenue_90d_per_acquired_customer': '€{:,.2f}',
    'purchaser_rate_90d': '{:.1%}',
})

**Decision:** Affiliate leads on 90-day revenue per acquired customer. Shift only marginal spend into a controlled test, because the dataset contains revenue but no media cost; CAC and incrementality remain guardrails.

## 3. Funnel leakage by channel

In [ ]:
funnel = pd.read_csv(PROCESSED / 'funnel_by_channel.csv')
funnel[['channel', 'view_to_cart_rate', 'cart_to_checkout_rate', 'checkout_to_order_rate', 'overall_conversion_rate']] \
    .sort_values('overall_conversion_rate', ascending=False) \
    .style.format('{:.1%}', subset=['view_to_cart_rate', 'cart_to_checkout_rate', 'checkout_to_order_rate', 'overall_conversion_rate'])

**Decision:** Social has the weakest checkout-to-order rate. Prioritize a mobile checkout experiment and estimate value with an explicit lift assumption instead of presenting correlation as causation.

## 4. RFM and churn risk

In [ ]:
segments = pd.read_csv(PROCESSED / 'segment_summary.csv')
segments.style.format({
    'net_revenue': '€{:,.0f}',
    'avg_customer_value': '€{:,.0f}',
    'churn_rate': '{:.1%}',
})

**Decision:** Treat At Risk as a value pool, not a blanket-discount audience. Rank by monetary value, personalize by last category, and use a holdout group to measure incremental reactivation.

## 5. Inspect the underlying SQL directly

In [ ]:
with sqlite3.connect(DB) as connection:
    top_products = pd.read_sql_query(
        """
        SELECT p.product_name, p.category,
               ROUND(SUM(op.allocated_net_revenue), 2) AS net_revenue,
               ROUND(SUM(op.allocated_gross_profit), 2) AS gross_profit
        FROM order_item_performance op
        JOIN products p USING (product_id)
        GROUP BY p.product_id, p.product_name, p.category
        ORDER BY net_revenue DESC
        LIMIT 10
        """, connection
    )
top_products

## 6. Recommendation discipline

Each recommendation in `reports/business_recommendations.md` contains: evidence, an action, an accountable owner, and an expected impact. The final step is experimental validation—especially for acquisition reallocation and funnel changes—because descriptive analytics alone does not establish causality.